# 00 — Project Overview: Laptop Price Prediction

> **Mục tiêu dự án:** Xây dựng mô hình dự đoán giá laptop dựa trên dữ liệu thu thập từ hai nguồn độc lập, sau đó hợp nhất và khai thác đặc trưng để huấn luyện mô hình hồi quy.

## Table of Contents

1. [Data Sources](#1-data-sources)
2. [Raw Dataset Snapshot](#2-raw-dataset-snapshot)
3. [Column Inventory](#3-column-inventory)
4. [Missing Value Summary](#4-missing-value-summary)
5. [Key Differences Between Sources](#5-key-differences-between-sources)
6. [Pipeline Roadmap](#6-pipeline-roadmap)
7. [Shared Conventions](#7-shared-conventions)

---
## 1. Data Sources

| # | Source | File | Nature |
|---|--------|------|--------|
| A | **Chợ Tốt** | `chotot_laptop_data.csv` | Thị trường mua bán đã qua sử dụng (secondhand marketplace) |
| B | **Websach** | `clean_laptop_features.csv` | Catalog laptop mới từ nhà bán lẻ, thu thập spec kỹ thuật đầy đủ |

Hai nguồn phản ánh **hai phân khúc thị trường khác nhau**: Chợ Tốt chủ yếu là laptop cũ do cá nhân đăng bán, trong khi Websach là sản phẩm mới đến từ các shop bán lẻ chính hãng. Việc kết hợp hai nguồn cho phép mô hình học được cả hai khoảng giá và đặc điểm sản phẩm đa dạng hơn.

---
## 2. Raw Dataset Snapshot

In [1]:
import pandas as pd

path_A = r"../../data/raw/chotot_laptop_data.csv"
path_B = r"../../data/raw/clean_laptop_features.csv"

df_A = pd.read_csv(path_A)
df_B = pd.read_csv(path_B)

In [2]:
summary = pd.DataFrame({
    "Source": ["Chợ Tốt (A)", "Websach (B)"],
    "Rows": [df_A.shape[0], df_B.shape[0]],
    "Columns": [df_A.shape[1], df_B.shape[1]],
    "Price column": ["price (string, VNĐ)", "shop_1/2/3_price (int, VNĐ)"],
    "Condition": ["Có cột Tình trạng (mới/cũ)", "Laptop mới — không có cột tình trạng"],
})
summary

,Source,Rows,Columns,Price column,Condition
0,Chợ Tốt (A),5866,15,"price (string, VNĐ)",Có cột Tình trạng (mới/cũ)
1,Websach (B),4384,43,"shop_1/2/3_price (int, VNĐ)",Laptop mới — không có cột tình trạng


---
## 3. Column Inventory

### 3A — Chợ Tốt (15 cột)

Dữ liệu tập trung vào **thông tin listing** (URL, tiêu đề, giá) và một số spec cơ bản mà người bán tự điền. Tất cả các cột đều ở dạng `object` — cần parse mạnh ở bước cleaning.

| Nhóm | Cột |
|------|-----|
| Metadata listing | `url`, `title`, `price` |
| Thông tin sản phẩm | `Hãng`, `Dòng máy`, `Tình trạng`, `Chính sách bảo hành`, `Xuất xứ`, `Thông tin sử dụng` |
| Spec kỹ thuật | `Kích cỡ màn hình`, `Bộ vi xử lý`, `RAM`, `Card màn hình`, `Ổ cứng`, `Loại ổ cứng` |

### 3B — Websach (43 cột)

Dữ liệu **spec kỹ thuật chi tiết** ở mức sâu hơn nhiều so với nguồn A. Giá được lấy từ tối đa 3 shop khác nhau cho cùng một model. Một số cột số (RAM, kích thước, tốc độ CPU…) đã được parse thành `float64`.

| Nhóm | Cột |
|------|-----|
| Nhận diện sản phẩm | `Hãng sản xuất`, `source_url` |
| CPU | `Công nghệ CPU`, `Loại CPU`, `Tốc độ CPU (GHz)`, `Tốc độ tối đa (GHz)` |
| RAM | `Loại RAM`, `Dung lượng RAM`, `Tốc độ bus`, `Hỗ trợ RAM tối đa` |
| Ổ cứng | `Loại ổ cứng`, `Dung lượng ổ cứng (GB)`, `Dung lượng` |
| Màn hình | `Công nghệ màn hình`, `Kích thước (inch)`, `Độ phân giải ngang (px)`, `Độ phân giải dọc (px)` |
| Card đồ họa | `Kiểu card đồ họa`, `Dung lượng VGA`, `Đồ họa đã làm sạch` |
| Vật lý | `Trọng lượng (kg)`, `Chiều dài (mm)`, `Chiều rộng (mm)`, `Độ dày (mm)`, `Chất liệu vỏ` |
| Kết nối & ngoại vi | `Cổng giao tiếp`, `Kết nối không dây`, `Webcam`, `Đèn bàn phím`, `Khe thẻ nhớ` |
| Hệ thống | `Hệ điều hành`, `Loại Pin`, `Công nghệ âm thanh`, `Tính năng khác` |
| Giá (multi-shop) | `shop_1_name/price/link`, `shop_2_name/price/link`, `shop_3_name/price/link` |

In [3]:
print("=== Chợ Tốt — Columns ===")
print(list(df_A.columns))

print("\n=== Websach — Columns ===")
print(list(df_B.columns))

=== Chợ Tốt — Columns ===
['url', 'price', 'title', 'Hãng', 'Dòng máy', 'Tình trạng', 'Chính sách bảo hành', 'Kích cỡ màn hình', 'Bộ vi xử lý', 'RAM', 'Card màn hình', 'Ổ cứng', 'Xuất xứ', 'Loại ổ cứng', 'Thông tin sử dụng']

=== Websach — Columns ===
['Hãng sản xuất', 'Hệ điều hành', 'Chất liệu vỏ', 'Công nghệ CPU', 'Loại CPU', 'Loại RAM', 'Dung lượng RAM', 'Tốc độ bus', 'Hỗ trợ RAM tối đa', 'Loại ổ cứng', 'Công nghệ màn hình', 'Kiểu card đồ họa', 'Công nghệ âm thanh', 'Cổng giao tiếp', 'Kết nối không dây', 'Webcam', 'Đèn bàn phím', 'Loại Pin', 'Dung lượng', 'source_url', 'shop_1_name', 'shop_1_price', 'shop_1_link', 'shop_2_name', 'shop_2_price', 'shop_2_link', 'shop_3_name', 'shop_3_price', 'shop_3_link', 'Khe thẻ nhớ', 'Tính năng khác', 'Dung lượng VGA', 'Dung lượng ổ cứng (GB)', 'Độ phân giải ngang (px)', 'Độ phân giải dọc (px)', 'Kích thước (inch)', 'Trọng lượng (kg)', 'Tốc độ CPU (GHz)', 'Tốc độ tối đa (GHz)', 'Chiều dài (mm)', 'Chiều rộng (mm)', 'Độ dày (mm)', 'Đồ họa đã làm sạ

---
## 4. Missing Value Summary

Cả hai nguồn đều có missing đáng kể. Đây là raw data nên **chưa xử lý gì** — các bước cleaning sẽ được thực hiện tại `02a` và `02b`.

In [4]:
def missing_report(df, name, top_n=10):
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(1)
    report = pd.DataFrame({"Missing": miss, "Missing %": miss_pct})
    report = report[report["Missing"] > 0].sort_values("Missing", ascending=False)
    print(f"\n=== {name} — {len(report)} cột có missing (top {top_n}) ===")
    return report.head(top_n)

display(missing_report(df_A, "Chợ Tốt"))
display(missing_report(df_B, "Websach"))


=== Chợ Tốt — 7 cột có missing (top 10) ===


,Missing,Missing %
Card màn hình,1421,24.2
Kích cỡ màn hình,1037,17.7
Loại ổ cứng,726,12.4
Ổ cứng,470,8.0
Bộ vi xử lý,249,4.2
RAM,239,4.1
price,1,0.0



=== Websach — 30 cột có missing (top 10) ===


,Missing,Missing %
Loại Pin,3036,69.3
Dung lượng VGA,2916,66.5
Hỗ trợ RAM tối đa,2474,56.4
Chất liệu vỏ,2215,50.5
Khe thẻ nhớ,2156,49.2
Dung lượng,650,14.8
Tốc độ CPU (GHz),635,14.5
Tốc độ bus,507,11.6
Độ dày (mm),456,10.4
Chiều dài (mm),456,10.4


**Nhận xét sơ bộ:**

- **Chợ Tốt:** Missing tập trung ở `Card màn hình` (~24%), `Kích cỡ màn hình` (~18%), `Loại ổ cứng` (~12%). Nguyên nhân chủ yếu do người bán không điền đủ thông tin khi đăng tin.
- **Websach:** Missing nghiêm trọng hơn ở các trường như `Loại Pin` (~69%), `Dung lượng VGA` (~67%), `Hỗ trợ RAM tối đa` (~56%), `Chất liệu vỏ` (~51%). Đây là các trường spec tùy chọn mà nhà sản xuất/shop không luôn công bố.
- Các outlier cực đoan trong giá và kích thước vật lý (e.g. `Trọng lượng` max 1112kg, `Kích thước` max 398 inch) cho thấy dữ liệu raw còn chứa **nhiễu** cần lọc.

---
## 5. Key Differences Between Sources

| Tiêu chí | Chợ Tốt (A) | Websach (B) |
|----------|-------------|-------------|
| **Phân khúc** | Secondhand — cá nhân đăng bán | Laptop mới — catalog nhà bán lẻ |
| **Số dòng** | 5,866 | 4,384 |
| **Số cột** | 15 | 43 |
| **Giá** | String (`"9.990.000 đ"`) — cần parse | Int (VNĐ) — đã sử dụng được, có 3 shop |
| **Khoảng giá** | Thị trường secondhand, đa dạng | Trung bình ~20–31 triệu (new retail) |
| **Độ chi tiết spec** | Thấp — người bán tự điền, dễ sai | Cao — crawl từ trang kỹ thuật chính thức |
| **Tình trạng máy** | Có (`Tình trạng` — mới/cũ/chưa sửa) | Không có (mặc định là mới) |
| **Dữ liệu số** | Tất cả object, cần extract | Nhiều cột đã parse thành float |
| **Outlier** | Giá lẻ, title tự do | Giá/kích thước bất thường cần lọc |
| **Unique identifier** | `url` | `source_url` |

---
## 6. Pipeline Roadmap

```
data/raw/
  ├── chotot_laptop_data.csv        ← nguồn A
  └── clean_laptop_features.csv     ← nguồn B
         │
         ▼
┌─────────────────────────────────────────────────┐
│  01a_chotot_inspection.ipynb                    │
│  01b_websach_inspection.ipynb                   │
│  → EDA chuyên sâu từng nguồn, xác định vấn đề   │
└─────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────┐
│  02a_chotot_cleaning.ipynb                      │
│  02b_websach_cleaning.ipynb                     │
│  → Parse giá, chuẩn hóa cột, xử lý missing,     │
│    loại outlier, export ra data/processed/      │
└─────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────┐
│  03_merge_and_eda.ipynb                         │
│  → Hợp nhất 2 nguồn, EDA tổng hợp, phân tích    │
│    phân phối giá, tương quan đặc trưng          │
└─────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────┐
│  04_feature_engineering.ipynb                   │
│  → Tạo đặc trưng mới, encode, scale, chuẩn bị   │
│    tập train/test cho modeling                  │
└─────────────────────────────────────────────────┘
```

| Notebook | Input | Output chính |
|----------|-------|--------------|
| `01a` | `chotot_laptop_data.csv` | Báo cáo EDA, danh sách vấn đề cần xử lý |
| `01b` | `clean_laptop_features.csv` | Báo cáo EDA, danh sách vấn đề cần xử lý |
| `02a` | `chotot_laptop_data.csv` | `processed/chotot_clean.csv` |
| `02b` | `clean_laptop_features.csv` | `processed/websach_clean.csv` |
| `03` | cả 2 processed | `processed/merged.csv` + insight EDA |
| `04` | `merged.csv` | `processed/features_ready.csv` |

---
## 7. Shared Conventions

Các quy ước thống nhất áp dụng xuyên suốt pipeline:

- **Đơn vị giá:** VNĐ (đồng), dạng số nguyên (int64) sau khi parse
- **Tên cột sau merge:** snake_case tiếng Anh, ví dụ `brand`, `ram_gb`, `storage_gb`, `screen_size_inch`, `price`
- **Nhãn nguồn:** thêm cột `source` với giá trị `'chotot'` hoặc `'websach'` trước khi merge
- **Outlier:** loại bằng ngưỡng IQR hoặc domain knowledge (ví dụ: giá < 1 triệu hoặc > 500 triệu bị drop)
- **Missing sau cleaning:** các cột thiếu > 50% sẽ được đánh giá lại tại `03_merge_and_eda` trước khi quyết định giữ hay bỏ
- **Random seed:** `42` cho tất cả các bước có yếu tố ngẫu nhiên